# 01 - Data Preprocessing & Feature Extraction
This notebook processes the raw time-series CNC force data and extracts statistical and frequency-domain features to generate the `FINAL_ML_DATASET.csv`.

**Pipeline:**
1. Load raw force/vibration datasets from `data/raw/`
2. Apply filtering and noise reduction (if necessary).
3. Chunk continuous cutting signals into discrete windows.
4. Extract Time-Domain features (`Fz_Mean`, `Fz_Std`, `Fz_Peak2Peak`).
5. Extract Frequency-Domain features (`Peak_FFT_Amp`).
6. Apply a rule-based labeling mechanism to label windows as Stable (0) or Chatter (1).
7. Save the processed dataset to `data/processed/`.


In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
import warnings

warnings.filterwarnings('ignore')
plt.style.use('dark_background')

### 1. Feature Extraction Functions
Define functions to extract both time and frequency domain features from a single 1D window of force data.


In [ ]:
def extract_time_features(window):
    """
    Extracts basic statistical time-domain features from a given window of force data.
    """
    fz_mean = np.mean(window)
    fz_std = np.std(window)
    fz_peak2peak = np.max(window) - np.min(window)
    return fz_mean, fz_std, fz_peak2peak

def extract_frequency_features(window, sampling_rate=10000):
    """
    Extracts the peak amplitude from the Fast Fourier Transform (FFT) of the signal.
    """
    N = len(window)
    # Perform FFT
    yf = fft(window)
    # Get only the positive frequencies
    yf_pos = np.abs(yf[:N//2])
    
    # We can ignore the DC component (0 Hz) by slicing from index 1
    if len(yf_pos) > 1:
        peak_fft_amp = np.max(yf_pos[1:])
    else:
        peak_fft_amp = 0.0
        
    return peak_fft_amp

### 2. Processing Pipeline
Iterate through the raw files, chunk them, extract features, and build the dataset.


In [ ]:
RAW_DATA_DIR = '../data/raw/'
PROCESSED_DATA_DIR = '../data/processed/'
OUTPUT_FILE = os.path.join(PROCESSED_DATA_DIR, 'FINAL_ML_DATASET.csv')
WINDOW_SIZE = 1024
SAMPLING_RATE = 10000 # Assume 10kHz sampling rate

raw_files = glob.glob(os.path.join(RAW_DATA_DIR, '*.csv'))
print(f"Found {len(raw_files)} raw CSV files.")

all_features = []
import re

# Mock threshold for rule-based labelling (example: if standard deviation exceeds X, it's chatter)
CHATTER_STD_THRESHOLD = 2.5 

for file in raw_files:
    # Parse RPM and DOC from filename (e.g., T1_1000doc_114RPM_05-02-2024.csv)
    basename = os.path.basename(file)
    doc_match = re.search(r'(\d+)doc', basename)
    rpm_match = re.search(r'(\d+)RPM', basename)
    
    if not doc_match or not rpm_match:
        print(f"Skipping {file}: Could not parse RPM or DOC from filename.")
        continue
        
    rpm = float(rpm_match.group(1))
    doc_um = float(doc_match.group(1))
    
    df = pd.read_csv(file)
    
    fz_col = 'Cutting Force - Fz (Newton)'
    if fz_col not in df.columns:
        print(f"Skipping {file}: Missing '{fz_col}' column.")
        continue
    
    signal = df[fz_col].values
    n_chunks = len(signal) // WINDOW_SIZE
    
    for i in range(n_chunks):
        start = i * WINDOW_SIZE
        end = start + WINDOW_SIZE
        window = signal[start:end]
        
        # Extract features
        fz_mean, fz_std, fz_p2p = extract_time_features(window)
        peak_fft = extract_frequency_features(window, SAMPLING_RATE)
        
        # Rule-based labelling (1 = Chatter, 0 = Stable)
        label = 1 if fz_std > CHATTER_STD_THRESHOLD else 0
        
        all_features.append({
            'RPM': rpm,
            'DOC_um': doc_um,
            'Fz_Mean': fz_mean,
            'Fz_Std': fz_std,
            'Fz_Peak2Peak': fz_p2p,
            'Peak_FFT_Amp': peak_fft,
            'Label': label
        })

if len(all_features) > 0:
    final_df = pd.DataFrame(all_features)
    final_df.to_csv(OUTPUT_FILE, index=False)
    print(f"\nSuccessfully processed {len(final_df)} windows.")
    print(f"Dataset saved to: {OUTPUT_FILE}")
    
    print("\nClass Distribution:")
    print(final_df['Label'].value_counts())
else:
    print("\nNo data processed.")


### 3. Quick Exploratory Data Analysis (EDA)
Visualize the resulting processed dataset to ensure the feature extraction worked correctly.


In [ ]:
if os.path.exists(OUTPUT_FILE):
    df_plot = pd.read_csv(OUTPUT_FILE)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Fz_Std Distribution
    df_plot[df_plot['Label'] == 0]['Fz_Std'].hist(ax=axes[0], alpha=0.7, bins=50, label='Stable', color='#00E5A0')
    df_plot[df_plot['Label'] == 1]['Fz_Std'].hist(ax=axes[0], alpha=0.7, bins=50, label='Chatter', color='#FF4C6E')
    axes[0].set_title('Distribution of Fz Standard Deviation')
    axes[0].set_xlabel('Fz_Std')
    axes[0].legend()
    
    # Peak FFT Amplitude Distribution
    df_plot[df_plot['Label'] == 0]['Peak_FFT_Amp'].hist(ax=axes[1], alpha=0.7, bins=50, label='Stable', color='#00E5A0')
    df_plot[df_plot['Label'] == 1]['Peak_FFT_Amp'].hist(ax=axes[1], alpha=0.7, bins=50, label='Chatter', color='#FF4C6E')
    axes[1].set_title('Distribution of Peak FFT Amplitude')
    axes[1].set_xlabel('Peak_FFT_Amp')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()